# Relayloom on a Kaggle GPU
Muse Glimmer 30B Q4_K_M (llama.cpp, CUDA) and SAM 3.1 (Ultralytics) over the repository's bills and photos. Real timings are saved to `/kaggle/working/gpu/`.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!nproc && free -g | head -2

In [ ]:
!pip -q install ultralytics huggingface_hub segno cryptography PyMuPDF
!rm -rf relayloom && git clone -q https://github.com/Sriharsan/relayloom.git && cd relayloom && git log --format='%h %an %s' -1

In [ ]:
# Build llama.cpp with CUDA for the Kaggle T4 (architecture 75). About ten minutes, once.
!git clone -q --depth 1 https://github.com/ggml-org/llama.cpp.git
!cd llama.cpp && cmake -B build -DGGML_CUDA=ON -DCMAKE_CUDA_ARCHITECTURES=75 -DLLAMA_CURL=OFF -DCMAKE_BUILD_TYPE=Release > /dev/null && cmake --build build --target llama-server -j 4 2>&1 | tail -2

In [ ]:
import os
from kaggle_secrets import UserSecretsClient
os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')  # your own token, from Kaggle secrets; never printed
from huggingface_hub import hf_hub_download
os.makedirs('/kaggle/working/models', exist_ok=True)
hf_hub_download('facebook/sam3.1', 'sam3.1_multiplex.pt', local_dir='/kaggle/working/models', token=os.environ['HF_TOKEN'])
for name in ('Muse-Glimmer-30B-KQuant-17GB-Q4_K_M.gguf', 'mmproj-Muse-Glimmer-30B-Q4_K_M.gguf'):
    hf_hub_download('meta-models/Muse-Glimmer-30B-GGUF', name, local_dir='/kaggle/working/models', token=os.environ['HF_TOKEN'])
print(sorted(os.listdir('/kaggle/working/models')))

In [ ]:
import subprocess, time, urllib.request
server = subprocess.Popen(['/kaggle/working/llama.cpp/build/bin/llama-server', '-m', '/kaggle/working/models/Muse-Glimmer-30B-KQuant-17GB-Q4_K_M.gguf',
    '--mmproj', '/kaggle/working/models/mmproj-Muse-Glimmer-30B-Q4_K_M.gguf', '--host', '127.0.0.1', '--port', '8084', '--jinja', '-c', '8192', '-np', '1',
    '-ngl', '99', '--split-mode', 'layer', '--reasoning-budget', '1024', '--chat-template-kwargs', '{"reasoning_strength":"low"}'],
    stdout=open('/kaggle/working/llama.log', 'w'), stderr=subprocess.STDOUT)
for _ in range(180):
    try:
        if urllib.request.urlopen('http://127.0.0.1:8084/health', timeout=3).status == 200: break
    except Exception: time.sleep(5)
print('Glimmer is up on both GPUs')

In [ ]:
import json, sys, time
os.chdir('/kaggle/working/relayloom'); sys.path.insert(0, '.')
os.environ['RELAYLOOM_MODELS'] = '/kaggle/working/models'
from relayloom.ap.models import GlimmerClient, SamCounter
from relayloom.ap.lookagent import LookAgent
ROUTE = 'Muse Glimmer 30B Q4_K_M and SAM 3.1, Kaggle GPU T4 x2'
glimmer = GlimmerClient('http://127.0.0.1:8084/v1', ROUTE, timeout=900)
sam = SamCounter(sys.executable, '/kaggle/working/models/sam3.1_multiplex.pt', ROUTE, live=True, cache=__import__('pathlib').Path('/kaggle/working/gpu/sam'))
os.makedirs('/kaggle/working/gpu', exist_ok=True)
# 1. Glimmer reads one bill, timed
from relayloom.ap.masking import mask
text = open('fixtures/bills/ARD-5101.txt', encoding='utf-8').read()
read = glimmer.read_bill(mask(text)[0])
print('bill read in', read['seconds'], 's, total', read['bill']['total_inr'])
bill_seconds = read['seconds']

In [ ]:
# 2. The agent loop on every photo that has a known answer: first try vs after the loop
import pathlib
photos = json.load(open('fixtures/data.json'))
truth = {p['id']: p for e in photos['receiving_events'] for p in e['photos']}
agent = LookAgent(glimmer, sam)
rows = []
for pid in sorted(truth)[:12]:
    path = pathlib.Path('fixtures/photos') / truth[pid]['file']
    event = next(e for e in photos['receiving_events'] if any(p['id'] == pid for p in e['photos']))
    item = photos['items'][event['item_code']]
    found = agent.look(path, item['name'], pid)
    rows.append({'photo': pid, 'agreed': found['agreed'], 'count': found['count'], 'first_try': found['first_try_count'], 'calls': found['calls'], 'seconds': found['seconds']})
    print(rows[-1])
json.dump({'route': ROUTE, 'bill_read_seconds': bill_seconds, 'rows': rows}, open('/kaggle/working/gpu/agent-loop.json', 'w'), indent=1)